# A_S1 — Synthetic Scatterplot Generation

Generates two datasets:

1. **Main dataset** (114,176 cases): 22 function families × SNR × spread × x_distribution,
   plus 128 Null cases (32 True Null + 96 Variance-only).
2. **Expanded Null dataset** (6,880 cases): additional replicates of Null cases
   for precise FPR calibration (4,000 True Null) and Variance-only detection analysis (2,880).

Model: `Y = f(X) + σ(X) · ε`

| f(X) | σ(X) | Category | Main | Expanded | Total |
|------|------|----------|-----:|---------:|------:|
| Constant | Constant | True Null | 32 | 4,000 | 4,032 |
| Varying | Constant | Mean-only | 28,512 | — | 28,512 |
| Constant | Varying | Variance-only | 96 | 2,880 | 2,976 |
| Varying | Varying | Mean+Variance | 85,536 | — | 85,536 |

In [ ]:
from __future__ import annotations

import json
import math
import time
import os
from dataclasses import dataclass, asdict
from itertools import product
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

os.environ.setdefault('MPLCONFIGDIR', str(Path('.matplotlib').resolve()))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

## Constants and Family Labels

In [ ]:
N_PER_CASE = 500
REPLICATES = 100
SIGNAL_VARIANCE_GRID_SIZE = 20_001

SNR_LEVELS = [0.1, 0.3, 0.5, 1, 2, 3, 5, 10, 20, 50, 100, math.inf]
SPREAD_PATTERNS = ['constant', 'increasing', 'decreasing', 'middle_high']
N_CLUSTERS_VALUES = [2, 3, 4, 5]
CLUSTER_OCCUPANCY = 0.4
CLUSTER_BACKGROUND_FRAC = 0.1
X_DISTRIBUTIONS = ['even', 'left_dense', 'right_dense', 'center_dense'] + [f'clusters_{k}' for k in N_CLUSTERS_VALUES]

# Expanded Null config
NULL_EXPAND_TRUE_NULL_PER_COMBO = 125    # 32 combos × 125 = 4,000
NULL_EXPAND_VAR_ONLY_PER_COMBO = 30      # 96 combos × 30  = 2,880
NULL_EXPAND_SEED_OFFSET = 500_000_000

NOISE_DISTRIBUTIONS = ['normal', 'uniform', 'heavy_tail', 'skewed']

OUTPUT_DIR = Path('output/S1')

FAMILY_LABELS: dict[str, dict[str, Any]] = {
    'F01': {
        'family_name': 'Linear positive',
        'equation_presentation': 'y = ax + b, a > 0',
        'basic_label': 'slope a',
        'direction': 'positive',
        'linearity': 'linear',
        'monotonicity': 'monotonic',
        'curvature': 'N/A',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F02': {
        'family_name': 'Linear negative',
        'equation_presentation': 'y = ax + b, a < 0',
        'basic_label': 'slope a',
        'direction': 'negative',
        'linearity': 'linear',
        'monotonicity': 'monotonic',
        'curvature': 'N/A',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F03': {
        'family_name': 'Power convex positive',
        'equation_presentation': 'y = x^p, p > 1',
        'basic_label': 'exponent p',
        'direction': 'positive',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'convex',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F04': {
        'family_name': 'Power convex negative',
        'equation_presentation': 'y = -x^p, p > 1',
        'basic_label': 'exponent p',
        'direction': 'negative',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'concave',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F05': {
        'family_name': 'Power concave positive',
        'equation_presentation': 'y = x^p, 0 < p < 1',
        'basic_label': 'exponent p',
        'direction': 'positive',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'concave',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F06': {
        'family_name': 'Power concave negative',
        'equation_presentation': 'y = -x^p, 0 < p < 1',
        'basic_label': 'exponent p',
        'direction': 'negative',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'convex',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F07': {
        'family_name': 'Saturation positive',
        'equation_presentation': 'y = 1 - exp(-kx)',
        'basic_label': 'rate k',
        'direction': 'positive',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'concave',
        'special_shape': 'saturation',
        'tp_number': '0',
    },
    'F08': {
        'family_name': 'Saturation negative',
        'equation_presentation': 'y = -(1 - exp(-kx))',
        'basic_label': 'rate k',
        'direction': 'negative',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'convex',
        'special_shape': 'saturation',
        'tp_number': '0',
    },
    'F09': {
        'family_name': 'Log positive',
        'equation_presentation': 'y = log(1 + ax)',
        'basic_label': 'scale a',
        'direction': 'positive',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'concave',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F10': {
        'family_name': 'Log negative',
        'equation_presentation': 'y = -log(1 + ax)',
        'basic_label': 'scale a',
        'direction': 'negative',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'convex',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F11': {
        'family_name': 'Exponential positive',
        'equation_presentation': 'y = a^x',
        'basic_label': 'base a, x range',
        'direction': 'positive',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'convex',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F12': {
        'family_name': 'Exponential negative',
        'equation_presentation': 'y = -a^x',
        'basic_label': 'base a, x range',
        'direction': 'negative',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'concave',
        'special_shape': 'none',
        'tp_number': '0',
    },
    'F13': {
        'family_name': 'S-curve positive',
        'equation_presentation': 'logistic (increasing)',
        'basic_label': 'steepness k, center c',
        'direction': 'positive',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'mixed',
        'special_shape': 'S-curve',
        'tp_number': '0',
    },
    'F14': {
        'family_name': 'S-curve negative',
        'equation_presentation': 'logistic (decreasing)',
        'basic_label': 'steepness k, center c',
        'direction': 'negative',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'mixed',
        'special_shape': 'S-curve',
        'tp_number': '0',
    },
    'F15': {
        'family_name': 'Threshold positive',
        'equation_presentation': 'piecewise, b > a',
        'basic_label': 'gap, width, position',
        'direction': 'positive',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'N/A',
        'special_shape': 'threshold',
        'tp_number': '0',
    },
    'F16': {
        'family_name': 'Threshold negative',
        'equation_presentation': 'piecewise, b < a',
        'basic_label': 'gap, width, position',
        'direction': 'negative',
        'linearity': 'nonlinear',
        'monotonicity': 'monotonic',
        'curvature': 'N/A',
        'special_shape': 'threshold',
        'tp_number': '0',
    },
    'F17': {
        'family_name': 'Quadratic peak (Inverted U-shape)',
        'equation_presentation': 'y = -a(x-c)^2 + d',
        'basic_label': 'curvature a, position c',
        'direction': 'none / local',
        'linearity': 'nonlinear',
        'monotonicity': 'non-monotonic',
        'curvature': 'N/A',
        'special_shape': 'peak (Inverted U-shape)',
        'tp_number': '1',
    },
    'F18': {
        'family_name': 'Quadratic valley (U-shape)',
        'equation_presentation': 'y = a(x-c)^2 + d',
        'basic_label': 'curvature a, position c',
        'direction': 'none / local',
        'linearity': 'nonlinear',
        'monotonicity': 'non-monotonic',
        'curvature': 'N/A',
        'special_shape': 'valley (U-shape)',
        'tp_number': '1',
    },
    'F19': {
        'family_name': 'Spike',
        'equation_presentation': 'narrow peak',
        'basic_label': 'width, position',
        'direction': 'none / local',
        'linearity': 'nonlinear',
        'monotonicity': 'non-monotonic',
        'curvature': 'N/A',
        'special_shape': 'local peak',
        'tp_number': '1',
    },
    'F20': {
        'family_name': 'L-shaped / narrow valley',
        'equation_presentation': 'piecewise valley',
        'basic_label': 'width, position',
        'direction': 'none / local',
        'linearity': 'nonlinear',
        'monotonicity': 'non-monotonic',
        'curvature': 'N/A',
        'special_shape': 'local valley',
        'tp_number': '1',
    },
    'F21': {
        'family_name': 'Cubic',
        'equation_presentation': 'cubic polynomial',
        'basic_label': 'roots, sign, spacing, type = M/W',
        'direction': 'none / mixed',
        'linearity': 'nonlinear',
        'monotonicity': 'non-monotonic',
        'curvature': 'mixed',
        'special_shape': 'cubic / two-turning',
        'tp_number': '2',
    },
    'F22': {
        'family_name': 'Complex non-monotonic functions',
        'equation_presentation': 'Double Gaussian, oscillation, etc.',
        'basic_label': 'shape-specific',
        'direction': 'none / mixed',
        'linearity': 'nonlinear',
        'monotonicity': 'non-monotonic',
        'curvature': 'mixed',
        'special_shape': 'complex',
        'tp_number': 'Multiple',
    },
    'Null': {
        'family_name': 'No relationship',
        'equation_presentation': 'noise only',
        'basic_label': 'none',
        'direction': 'N/A',
        'linearity': 'N/A',
        'monotonicity': 'N/A',
        'curvature': 'N/A',
        'special_shape': 'no relationship',
        'tp_number': 'N/A',
    },
}

label_df = pd.DataFrame.from_dict(FAMILY_LABELS, orient='index').reset_index(names='family_id')
display(label_df)

## Shape Configuration Library

In [ ]:
@dataclass(frozen=True)
class ShapeConfig:
    shape_config_id: int
    family_id: str
    variant_name: str
    params: dict[str, Any]

    @property
    def labels(self) -> dict[str, Any]:
        return FAMILY_LABELS[self.family_id]


def _add_config(configs: list[ShapeConfig], family_id: str, variant_name: str, **params: Any) -> None:
    configs.append(
        ShapeConfig(
            shape_config_id=len(configs) + 1,
            family_id=family_id,
            variant_name=variant_name,
            params=params,
        )
    )


def build_shape_configs() -> list[ShapeConfig]:
    configs: list[ShapeConfig] = []

    for slope_abs in [0.1, 0.3, 0.5, 1, 2, 5]:
        _add_config(configs, 'F01', f'slope_{slope_abs:g}', slope=float(slope_abs), intercept=0.0)
        _add_config(configs, 'F02', f'slope_-{slope_abs:g}', slope=-float(slope_abs), intercept=0.0)

    for p in [1.5, 2, 3, 5]:
        _add_config(configs, 'F03', f'p_{p:g}', p=float(p))
        _add_config(configs, 'F04', f'p_{p:g}', p=float(p))

    for p in [0.2, 0.3, 0.5, 0.7]:
        _add_config(configs, 'F05', f'p_{p:g}', p=float(p))
        _add_config(configs, 'F06', f'p_{p:g}', p=float(p))

    for k in [0.5, 1, 2, 3, 5, 8, 10, 15, 20]:
        _add_config(configs, 'F07', f'k_{k:g}', k=float(k))
        _add_config(configs, 'F08', f'k_{k:g}', k=float(k))

    for a in [1, 5, 10, 20, 50]:
        _add_config(configs, 'F09', f'a_{a:g}', a=float(a))
        _add_config(configs, 'F10', f'a_{a:g}', a=float(a))

    for base, x_range in product([1.5, 2, 5, 10], [1, 3, 5]):
        _add_config(configs, 'F11', f'base_{base:g}_range_{x_range:g}', base=float(base), x_range=float(x_range))
        _add_config(configs, 'F12', f'base_{base:g}_range_{x_range:g}', base=float(base), x_range=float(x_range))

    for k, c in product([5, 8, 12, 20, 30], [0.3, 0.5, 0.7]):
        _add_config(configs, 'F13', f'k_{k:g}_c_{c:g}', k=float(k), c=float(c), L1=0.0, L2=1.0)
        _add_config(configs, 'F14', f'k_{k:g}_c_{c:g}', k=float(k), c=float(c), L1=0.0, L2=1.0)

    # F15/F16: 3 gap × 3 width × 2 position = 18 each
    for gap, delta, c in product([0.3, 1, 2], [0.005, 0.03, 0.05], [0.3, 0.7]):
        _add_config(configs, 'F15', f'gap_{gap:g}_delta_{delta:g}_c_{c:g}', a=0.0, b=float(gap), delta=float(delta), c=float(c))
        _add_config(configs, 'F16', f'gap_{gap:g}_delta_{delta:g}_c_{c:g}', a=0.0, b=float(gap), delta=float(delta), c=float(c))

    for curvature, c in product([1, 4, 10, 20], [0.2, 0.35, 0.5, 0.65, 0.8]):
        _add_config(configs, 'F17', f'a_{curvature:g}_c_{c:g}', a=float(curvature), c=float(c), d=0.0)
        _add_config(configs, 'F18', f'a_{curvature:g}_c_{c:g}', a=float(curvature), c=float(c), d=0.0)

    for width, c in product([0.02, 0.05, 0.1, 0.2], [0.3, 0.5, 0.7]):
        _add_config(configs, 'F19', f'width_{width:g}_c_{c:g}', width=float(width), c=float(c))
        _add_config(configs, 'F20', f'width_{width:g}_c_{c:g}', width=float(width), c=float(c))

    cubic_positions = {'left': 0.35, 'center': 0.5, 'right': 0.65}
    cubic_spacings = {
        'narrow': {'half_width': 0.15, 'a_abs': 30.0},
        'medium': {'half_width': 0.25, 'a_abs': 18.0},
        'wide':   {'half_width': 0.35, 'a_abs': 10.0},
    }
    for type_name, sign in [('m', 1.0), ('w', -1.0)]:
        for position_name, center in cubic_positions.items():
            for spacing_name, sp in cubic_spacings.items():
                hw = sp['half_width']
                roots = [center - hw, center, center + hw]
                _add_config(configs, 'F21', f'cubic_{type_name}_{position_name}_{spacing_name}',
                            roots=[float(r) for r in roots], a=float(sign * sp['a_abs']), d=0.0,
                            cubic_type='M' if type_name == 'm' else 'W',
                            position=position_name, spacing=spacing_name)

    complex_variants = [
        ('double_gaussian_equal', dict(A1=1.0, mu1=0.3, sigma1=0.08, A2=1.0, mu2=0.7, sigma2=0.08)),
        ('double_gaussian_unequal', dict(A1=1.0, mu1=0.3, sigma1=0.07, A2=0.55, mu2=0.72, sigma2=0.11)),
        ('pure_osc_low', dict(A=1.0, omega=2*math.pi)),
        ('pure_osc_high', dict(A=1.0, omega=8*math.pi)),
        ('oscillation_trend_weak', dict(b=0.4, A=0.4, omega=4*math.pi)),
        ('oscillation_trend_strong', dict(b=1.2, A=0.25, omega=4*math.pi)),
        ('damped_oscillation', dict(A=1.0, **{'lambda': 2.0}, omega=8*math.pi)),
        ('growing_oscillation', dict(A=0.25, **{'lambda': 2.0}, omega=8*math.pi)),
        ('varying_frequency', dict(A=1.0, omega=6*math.pi)),
    ]
    for name, params in complex_variants:
        _add_config(configs, 'F22', name, **params)

    _add_config(configs, 'Null', 'independent_normal', noise_dist='normal')
    _add_config(configs, 'Null', 'independent_uniform', noise_dist='uniform')
    _add_config(configs, 'Null', 'independent_heavy_tail', noise_dist='heavy_tail')
    _add_config(configs, 'Null', 'independent_skewed', noise_dist='skewed')
    return configs


shape_configs = build_shape_configs()
shape_df = pd.DataFrame([
    {**asdict(cfg), **cfg.labels, 'params_json': json.dumps(cfg.params, sort_keys=True)}
    for cfg in shape_configs
]).drop(columns=['params'])

display(shape_df.groupby('family_id', sort=False).size().rename('n_configs').reset_index())
print(f'Shape configs: {len(shape_configs)}')

## Function Evaluation

In [ ]:
_NEGATIVE_COUNTERPART = {
    'F04': 'F03', 'F06': 'F05', 'F08': 'F07', 'F10': 'F09',
    'F12': 'F11', 'F14': 'F13', 'F16': 'F15',
}


def _triangular_local(x: np.ndarray, center: float, width: float) -> np.ndarray:
    return np.maximum(0.0, 1.0 - np.abs(x - center) / width)


def evaluate_signal(x: np.ndarray, cfg: ShapeConfig) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    p = cfg.params
    fid = cfg.family_id

    if fid in _NEGATIVE_COUNTERPART:
        positive_cfg = ShapeConfig(
            shape_config_id=cfg.shape_config_id,
            family_id=_NEGATIVE_COUNTERPART[fid],
            variant_name=cfg.variant_name,
            params=cfg.params,
        )
        return -evaluate_signal(x, positive_cfg)

    if fid in {'F01', 'F02'}:
        return p['slope'] * x + p.get('intercept', 0.0)
    if fid in {'F03', 'F05'}:
        return x ** p['p']
    if fid == 'F07':
        return 1.0 - np.exp(-p['k'] * x)
    if fid == 'F09':
        return np.log1p(p['a'] * x)
    if fid == 'F11':
        return p['base'] ** (x * p['x_range'])
    if fid == 'F13':
        return p['L1'] + (p['L2'] - p['L1']) / (1.0 + np.exp(-p['k'] * (x - p['c'])))
    if fid == 'F15':
        t = np.clip((x - (p['c'] - p['delta'])) / (2.0 * p['delta']), 0.0, 1.0)
        return p['a'] + (p['b'] - p['a']) * t
    if fid == 'F17':
        return -p['a'] * (x - p['c']) ** 2 + p['d']
    if fid == 'F18':
        return p['a'] * (x - p['c']) ** 2 + p['d']
    if fid == 'F19':
        return _triangular_local(x, p['c'], p['width'])
    if fid == 'F20':
        return 1.0 - _triangular_local(x, p['c'], p['width'])
    if fid == 'F21':
        r1, r2, r3 = p['roots']
        return p['a'] * (x - r1) * (x - r2) * (x - r3) + p['d']
    if fid == 'F22':
        name = cfg.variant_name
        if name.startswith('double_gaussian'):
            return (p['A1'] * np.exp(-((x - p['mu1'])**2) / p['sigma1']**2)
                  + p['A2'] * np.exp(-((x - p['mu2'])**2) / p['sigma2']**2))
        if name.startswith('pure_osc'):
            return p['A'] * np.sin(p['omega'] * x)
        if name.startswith('oscillation_trend'):
            return p['b'] * x + p['A'] * np.sin(p['omega'] * x)
        if name == 'damped_oscillation':
            return p['A'] * np.exp(-p['lambda'] * x) * np.sin(p['omega'] * x)
        if name == 'growing_oscillation':
            return p['A'] * np.exp(p['lambda'] * x) * np.sin(p['omega'] * x)
        if name == 'varying_frequency':
            return p['A'] * np.sin(p['omega'] * x * (1.0 + x))
        raise ValueError(f'Unknown F22 variant: {name}')
    if fid == 'Null':
        return np.zeros_like(x, dtype=float)

    raise ValueError(f'Unknown family_id: {fid}')


def get_shape_config(shape_config_id: int) -> ShapeConfig:
    return shape_configs[shape_config_id - 1]

## Sampling and Noise

In [ ]:
def _cluster_bounds(n_clusters: int) -> list[tuple[float, float]]:
    cluster_width = CLUSTER_OCCUPANCY / n_clusters
    gap_width = (1.0 - CLUSTER_OCCUPANCY) / (n_clusters + 1)
    bounds = []
    for i in range(n_clusters):
        lo = gap_width * (i + 1) + cluster_width * i
        hi = lo + cluster_width
        bounds.append((lo, hi))
    return bounds


def sample_x(n: int, distribution: str, rng: np.random.Generator) -> np.ndarray:
    if distribution == 'even':
        return rng.uniform(0.0, 1.0, size=n)
    if distribution == 'left_dense':
        return rng.beta(2.0, 5.0, size=n)
    if distribution == 'right_dense':
        return rng.beta(5.0, 2.0, size=n)
    if distribution == 'center_dense':
        return rng.beta(5.0, 5.0, size=n)
    if distribution.startswith('clusters_'):
        n_clusters = int(distribution.split('_')[1])
        bounds = _cluster_bounds(n_clusters)
        is_background = rng.random(size=n) < CLUSTER_BACKGROUND_FRAC
        n_bg = is_background.sum()
        component = rng.integers(0, n_clusters, size=n)
        x = np.empty(n, dtype=float)
        for i, (lo, hi) in enumerate(bounds):
            mask = (~is_background) & (component == i)
            x[mask] = rng.uniform(lo, hi, size=mask.sum())
        x[is_background] = rng.uniform(0.0, 1.0, size=n_bg)
        return x
    raise ValueError(f'Unknown x distribution: {distribution}')


def spread_multiplier(x: np.ndarray, pattern: str) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if pattern == 'constant':
        return np.ones_like(x)
    if pattern == 'increasing':
        return 0.2 + 1.6 * x
    if pattern == 'decreasing':
        return 1.8 - 1.6 * x
    if pattern == 'middle_high':
        return 0.3 + 1.4 * np.exp(-((x - 0.5)**2) / 0.02)
    raise ValueError(f'Unknown spread pattern: {pattern}')


_signal_variance_cache: dict[int, float] = {}

def signal_variance_under_uniform(cfg: ShapeConfig) -> float:
    if cfg.shape_config_id not in _signal_variance_cache:
        if cfg.family_id == 'Null':
            variance = 0.0
        else:
            x_grid = np.linspace(0.0, 1.0, SIGNAL_VARIANCE_GRID_SIZE)
            variance = float(np.var(evaluate_signal(x_grid, cfg), ddof=0))
        _signal_variance_cache[cfg.shape_config_id] = variance
    return _signal_variance_cache[cfg.shape_config_id]


def sigma0_for_snr(cfg: ShapeConfig, snr: float) -> float:
    if math.isinf(float(snr)):
        return 0.0
    variance = signal_variance_under_uniform(cfg)
    if variance <= 0.0:
        return 0.0
    return math.sqrt(variance / float(snr))


X_DIST_LABELS: dict[str, dict[str, str]] = {
    'even': {'density_label': 'even', 'cluster_label': 'no clusters'},
    'left_dense': {'density_label': 'uneven, left-dense', 'cluster_label': 'no clusters'},
    'right_dense': {'density_label': 'uneven, right-dense', 'cluster_label': 'no clusters'},
    'center_dense': {'density_label': 'uneven, center-dense', 'cluster_label': 'no clusters'},
}

def _x_dist_labels(x_distribution: str) -> dict[str, str]:
    if x_distribution in X_DIST_LABELS:
        return X_DIST_LABELS[x_distribution]
    if x_distribution.startswith('clusters_'):
        return {'density_label': 'uneven', 'cluster_label': 'clusters'}
    raise ValueError(f'Unknown x distribution: {x_distribution}')

## Build Case Metadata

In [ ]:
def build_cases() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    def add_case(cfg: ShapeConfig, snr: float, spread_pattern: str, x_distribution: str) -> None:
        labels = cfg.labels
        x_labels = _x_dist_labels(x_distribution)
        rows.append({
            'case_id': len(rows) + 1,
            'shape_config_id': cfg.shape_config_id,
            'family_id': cfg.family_id,
            'family_name': labels['family_name'],
            'variant_name': cfg.variant_name,
            'params_json': json.dumps(cfg.params, sort_keys=True),
            'snr': 'inf' if math.isinf(float(snr)) else float(snr),
            'spread_pattern': spread_pattern,
            'x_distribution': x_distribution,
            'n_clusters': int(x_distribution.split('_')[1]) if x_distribution.startswith('clusters_') else 0,
            'density_label': x_labels['density_label'],
            'cluster_label': x_labels['cluster_label'],
            'n': N_PER_CASE,
            'replicates': REPLICATES,
            **{k: v for k, v in labels.items() if k != 'family_name'},
        })

    for cfg in shape_configs:
        if cfg.family_id == 'Null':
            for spread, x_dist in product(SPREAD_PATTERNS, X_DISTRIBUTIONS):
                add_case(cfg, math.inf, spread, x_dist)
        else:
            for snr, spread, x_dist in product(SNR_LEVELS, SPREAD_PATTERNS, X_DISTRIBUTIONS):
                add_case(cfg, snr, spread, x_dist)

    return pd.DataFrame(rows)


cases_df = build_cases()

is_null = cases_df['family_id'] == 'Null'
is_const = cases_df['spread_pattern'] == 'constant'
cases_df['category'] = 'mean+variance'
cases_df.loc[is_null & is_const, 'category'] = 'true_null'
cases_df.loc[~is_null & is_const, 'category'] = 'mean_only'
cases_df.loc[is_null & ~is_const, 'category'] = 'variance_only'

print(f'Total cases: {len(cases_df):,}')
print()
print(cases_df['category'].value_counts())

## Generate Scatterplot Points

In [ ]:
def generate_scatter_xy(case_id: int, replicate: int = 0,
                        cases: pd.DataFrame | None = None,
                        seed_offset: int = 0) -> tuple[np.ndarray, np.ndarray]:
    """Generate (x, y) arrays for a single case. Lightweight version for export."""
    if cases is None:
        cases = cases_df
    case = cases.loc[cases['case_id'] == case_id].iloc[0]
    cfg = get_shape_config(int(case['shape_config_id']))
    seed = seed_offset + int(case_id) * 1000 + int(replicate)
    rng = np.random.default_rng(seed)

    x = sample_x(N_PER_CASE, str(case['x_distribution']), rng)
    y_signal = evaluate_signal(x, cfg)

    if cfg.family_id == 'Null':
        noise_dist = cfg.params.get('noise_dist', 'normal')
        sigma_i = 1.0 * spread_multiplier(x, str(case['spread_pattern']))
        if noise_dist == 'normal':
            raw = rng.normal(0.0, 1.0, size=N_PER_CASE)
        elif noise_dist == 'uniform':
            raw = rng.uniform(-1.0, 1.0, size=N_PER_CASE) * np.sqrt(3.0)
        elif noise_dist == 'heavy_tail':
            raw = rng.standard_t(df=3, size=N_PER_CASE) / np.sqrt(3.0)
        elif noise_dist == 'skewed':
            raw = rng.exponential(1.0, size=N_PER_CASE) - 1.0
        else:
            raise ValueError(f'Unknown noise distribution: {noise_dist}')
        y = raw * sigma_i
    else:
        snr = math.inf if str(case['snr']) == 'inf' else float(case['snr'])
        sigma0 = sigma0_for_snr(cfg, snr)
        sigma_i = sigma0 * spread_multiplier(x, str(case['spread_pattern']))
        y = y_signal + rng.normal(0.0, sigma_i, size=N_PER_CASE)

    return x.astype(np.float32), y.astype(np.float32)

## Export 1: Main Dataset (114,176 cases)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
n_main = len(cases_df)

cases_df.to_csv(OUTPUT_DIR / 'cases.csv', index=False)
print(f'Saved cases.csv ({n_main:,} rows)')

x_main = np.empty((n_main, N_PER_CASE), dtype=np.float32)
y_main = np.empty((n_main, N_PER_CASE), dtype=np.float32)

t0 = time.time()
for idx in range(n_main):
    cid = int(cases_df.iloc[idx]['case_id'])
    x_main[idx], y_main[idx] = generate_scatter_xy(cid)
    if (idx + 1) % 10000 == 0:
        elapsed = time.time() - t0
        rate = (idx + 1) / elapsed
        eta = (n_main - idx - 1) / rate
        print(f'  {idx+1:>7,}/{n_main:,}  ({rate:.0f} cases/s, ETA {eta:.0f}s)')

np.savez_compressed(OUTPUT_DIR / 'scatter_points.npz', x=x_main, y=y_main)
elapsed = time.time() - t0
size_mb = (OUTPUT_DIR / 'scatter_points.npz').stat().st_size / 1e6
print(f'Saved scatter_points.npz  shape=({n_main:,}, {N_PER_CASE})  {size_mb:.1f} MB  ({elapsed:.0f}s)')

## Export 2: Expanded Null Dataset (6,880 cases)

Same 128 parameter combinations as the main dataset's Null cases,
but with many more replicates per combination (different random seeds).

- True Null (constant spread): 32 combos × 125 replicates = 4,000
- Variance-only (non-constant spread): 96 combos × 30 replicates = 2,880

In [ ]:
def build_null_expanded_cases() -> pd.DataFrame:
    null_configs = [cfg for cfg in shape_configs if cfg.family_id == 'Null']
    rows: list[dict[str, Any]] = []

    for cfg in null_configs:
        for spread in SPREAD_PATTERNS:
            for x_dist in X_DISTRIBUTIONS:
                is_true_null = (spread == 'constant')
                n_reps = NULL_EXPAND_TRUE_NULL_PER_COMBO if is_true_null else NULL_EXPAND_VAR_ONLY_PER_COMBO
                category = 'true_null' if is_true_null else 'variance_only'
                labels = cfg.labels
                x_labels = _x_dist_labels(x_dist)

                for k in range(n_reps):
                    rows.append({
                        'case_id': len(rows) + 1,
                        'shape_config_id': cfg.shape_config_id,
                        'family_id': 'Null',
                        'family_name': labels['family_name'],
                        'variant_name': cfg.variant_name,
                        'params_json': json.dumps(cfg.params, sort_keys=True),
                        'snr': 'inf',
                        'spread_pattern': spread,
                        'x_distribution': x_dist,
                        'n_clusters': int(x_dist.split('_')[1]) if x_dist.startswith('clusters_') else 0,
                        'density_label': x_labels['density_label'],
                        'cluster_label': x_labels['cluster_label'],
                        'n': N_PER_CASE,
                        'replicates': 1,
                        'replicate_id': k,
                        'category': category,
                        **{kk: v for kk, v in labels.items() if kk != 'family_name'},
                    })

    return pd.DataFrame(rows)


null_exp_df = build_null_expanded_cases()
n_null_exp = len(null_exp_df)

print(f'Expanded Null cases: {n_null_exp:,}')
print(null_exp_df['category'].value_counts())
print()
print(f'True Null combos: {(null_exp_df["category"]=="true_null").sum()}')
print(f'Variance-only combos: {(null_exp_df["category"]=="variance_only").sum()}')

In [ ]:
null_exp_df.to_csv(OUTPUT_DIR / 'null_expanded_cases.csv', index=False)
print(f'Saved null_expanded_cases.csv ({n_null_exp:,} rows)')

x_null_exp = np.empty((n_null_exp, N_PER_CASE), dtype=np.float32)
y_null_exp = np.empty((n_null_exp, N_PER_CASE), dtype=np.float32)

t0 = time.time()
for idx in range(n_null_exp):
    cid = int(null_exp_df.iloc[idx]['case_id'])
    x_null_exp[idx], y_null_exp[idx] = generate_scatter_xy(
        cid, cases=null_exp_df, seed_offset=NULL_EXPAND_SEED_OFFSET)
    if (idx + 1) % 1000 == 0:
        elapsed = time.time() - t0
        rate = (idx + 1) / elapsed
        print(f'  {idx+1:>6,}/{n_null_exp:,}  ({rate:.0f} cases/s)')

np.savez_compressed(OUTPUT_DIR / 'null_expanded_points.npz', x=x_null_exp, y=y_null_exp)
elapsed = time.time() - t0
size_mb = (OUTPUT_DIR / 'null_expanded_points.npz').stat().st_size / 1e6
print(f'Saved null_expanded_points.npz  shape=({n_null_exp:,}, {N_PER_CASE})  {size_mb:.1f} MB  ({elapsed:.0f}s)')

## Verification

In [ ]:
print('=== Main Dataset ===')
print(f'  cases.csv:          {len(cases_df):>10,} rows')
print(f'  scatter_points.npz: {x_main.shape}')
print(f'  Categories:')
print(cases_df['category'].value_counts().to_string(header=False))
print()
print('=== Expanded Null ===')
print(f'  null_expanded_cases.csv:  {n_null_exp:>10,} rows')
print(f'  null_expanded_points.npz: {x_null_exp.shape}')
print(f'  Categories:')
print(null_exp_df['category'].value_counts().to_string(header=False))
print()
print('=== Combined Totals ===')
print(f'  True Null:      {(cases_df["category"]=="true_null").sum():>6,} + {(null_exp_df["category"]=="true_null").sum():>5,} = {(cases_df["category"]=="true_null").sum() + (null_exp_df["category"]=="true_null").sum():>6,}')
print(f'  Variance-only:  {(cases_df["category"]=="variance_only").sum():>6,} + {(null_exp_df["category"]=="variance_only").sum():>5,} = {(cases_df["category"]=="variance_only").sum() + (null_exp_df["category"]=="variance_only").sum():>6,}')
print(f'  Mean-only:      {(cases_df["category"]=="mean_only").sum():>6,}')
print(f'  Mean+Variance:  {(cases_df["category"]=="mean+variance").sum():>6,}')

In [ ]:
# Spot-check: reproduce a main case
test_idx = 100
test_cid = int(cases_df.iloc[test_idx]['case_id'])
x_t, y_t = generate_scatter_xy(test_cid)
assert np.allclose(x_main[test_idx], x_t), 'Main dataset reproducibility failed'
assert np.allclose(y_main[test_idx], y_t), 'Main dataset reproducibility failed'

# Spot-check: reproduce a null expanded case
test_idx_n = 50
test_cid_n = int(null_exp_df.iloc[test_idx_n]['case_id'])
x_tn, y_tn = generate_scatter_xy(test_cid_n, cases=null_exp_df, seed_offset=NULL_EXPAND_SEED_OFFSET)
assert np.allclose(x_null_exp[test_idx_n], x_tn), 'Null expanded reproducibility failed'
assert np.allclose(y_null_exp[test_idx_n], y_tn), 'Null expanded reproducibility failed'

print('Reproducibility checks passed')